In [1]:
import os
import glob
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, event, text
from sqlalchemy.engine import URL

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'

RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(ROOT, 'reports')
DB_PATH = os.path.join(PROCESSED_DIR, 'bluestock_dw.db')

print('Database file exists:', os.path.exists(DB_PATH))

engine = create_engine(URL.create('sqlite', database=DB_PATH))

@event.listens_for(engine, 'connect')
def turn_on_foreign_keys(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute('PRAGMA foreign_keys = ON')
    cursor.close()

Database file exists: True


In [2]:
with engine.connect() as conn:
    print('Foreign keys on (1 = yes):', conn.execute(text('PRAGMA foreign_keys')).scalar())

pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", engine)

Foreign keys on (1 = yes): 1


,name
0,dim_date
1,dim_fund
2,fact_aum
3,fact_nav
4,fact_performance
5,fact_transactions


In [3]:
def to_date_key(dates):
    return dates.dt.strftime('%Y%m%d').astype(int)


def build_dim_date(start, end):
    day = pd.Series(pd.date_range(start, end, freq='D'))
    return pd.DataFrame({
        'date_key': day.dt.strftime('%Y%m%d').astype(int),
        'full_date': day.dt.strftime('%Y-%m-%d'),
        'year': day.dt.year,
        'quarter': day.dt.quarter,
        'month': day.dt.month,
        'month_name': day.dt.month_name(),
        'day_of_month': day.dt.day,
        'day_of_week': day.dt.dayofweek,
        'day_name': day.dt.day_name(),
        'week_of_year': day.dt.isocalendar().week.astype(int),
        'is_weekend': (day.dt.dayofweek >= 5).astype(int),
        'fiscal_year': np.where(day.dt.month >= 4, day.dt.year + 1, day.dt.year),
        'fiscal_quarter': ((day.dt.month - 4) % 12) // 3 + 1,
    })

In [4]:
build_dim_date('2026-10-09', '2026-10-09').iloc[0]

date_key            20261009
full_date         2026-10-09
year                    2026
quarter                    4
month                     10
month_name           October
day_of_month               9
day_of_week                4
day_name              Friday
week_of_year              41
is_weekend                 0
fiscal_year             2027
fiscal_quarter             3
Name: 0, dtype: object

In [7]:
master_path = glob.glob(os.path.join(RAW_DIR, '*01_fund_master*'))[0]
master = pd.read_csv(master_path)

DIM_FUND_COLS = ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
                 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct',
                 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category',
                 'sebi_category_code']

dim_fund = master[DIM_FUND_COLS].copy()

for col in ['fund_house', 'scheme_name', 'category', 'sub_category', 'plan',
            'benchmark', 'fund_manager', 'risk_category']:
    dim_fund[col] = dim_fund[col].str.strip()

dim_fund['sebi_category_code'] = dim_fund['sebi_category_code'].astype('string')

print(master['launch_date'].head())

0    2006-02-14
1    2013-01-01
2    2009-09-09
3    2013-01-01
4    2000-12-30
Name: launch_date, dtype: str


In [8]:
LAUNCH_DATE_FORMAT = '%Y-%m-%d'

launch = pd.to_datetime(dim_fund['launch_date'], format=LAUNCH_DATE_FORMAT, errors='coerce')
print('launch_date not readable:', (launch.isnull() & dim_fund['launch_date'].notnull()).sum())

dim_fund['launch_date'] = launch.dt.strftime('%Y-%m-%d')

launch_date not readable: 0


In [9]:
print('Rows:', len(dim_fund))
print('Missing amfi_code:', dim_fund['amfi_code'].isnull().sum())
print('Repeated amfi_code:', dim_fund['amfi_code'].duplicated().sum())
print('Missing fund_house:', dim_fund['fund_house'].isnull().sum())
print('Missing scheme_name:', dim_fund['scheme_name'].isnull().sum())

for col in ['expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount']:
    print(col, 'below 0:', (dim_fund[col] < 0).sum())

Rows: 40
Missing amfi_code: 0
Repeated amfi_code: 0
Missing fund_house: 0
Missing scheme_name: 0
expense_ratio_pct below 0: 0
exit_load_pct below 0: 0
min_sip_amount below 0: 0
min_lumpsum_amount below 0: 0


In [10]:
nav_path = os.path.join(PROCESSED_DIR, 'nav_history_clean.csv')
nav = pd.read_csv(nav_path, parse_dates=['date'])

fact_nav = pd.DataFrame({
    'amfi_code': nav['amfi_code'],
    'date_key': to_date_key(nav['date']),
    'nav': nav['nav'],
    'is_filled': nav['is_filled'].astype(int),
})

print(fact_nav.shape)
print('Repeated (amfi_code, date_key):', fact_nav.duplicated(subset=['amfi_code', 'date_key']).sum())
print('NAV zero or below:', (fact_nav['nav'] <= 0).sum())

(64320, 4)
Repeated (amfi_code, date_key): 0
NAV zero or below: 0


In [11]:
tx_path = os.path.join(PROCESSED_DIR, 'investor_transactions_clean.csv')
tx = pd.read_csv(tx_path, parse_dates=['transaction_date'])

fact_transactions = pd.DataFrame({
    'investor_id': tx['investor_id'].astype('string'),
    'date_key': to_date_key(tx['transaction_date']),
    'amfi_code': tx['amfi_code'],
    'transaction_type': tx['transaction_type'],
    'amount_inr': tx['amount_inr'],
    'payment_mode': tx['payment_mode'],
    'kyc_status': tx['kyc_status'],
    'state': tx['state'],
    'city': tx['city'],
    'city_tier': tx['city_tier'].astype('string'),
    'age_group': tx['age_group'],
    'gender': tx['gender'],
    'annual_income_lakh': tx['annual_income_lakh'],
})

print(fact_transactions.shape)
print(fact_transactions['transaction_type'].value_counts())
print('Amount zero or below:', (fact_transactions['amount_inr'] <= 0).sum())

(32778, 13)
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64
Amount zero or below: 0


In [12]:
perf_path = os.path.join(PROCESSED_DIR, 'scheme_performance_clean.csv')
perf = pd.read_csv(perf_path)

SNAPSHOT_DATE = nav['date'].max()
SNAPSHOT_KEY = int(SNAPSHOT_DATE.strftime('%Y%m%d'))
print('Snapshot date:', SNAPSHOT_DATE.date())

PERF_COLS = ['amfi_code', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
             'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
             'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
             'morningstar_rating', 'anomaly_count']

fact_performance = perf[PERF_COLS].copy()
fact_performance.insert(1, 'snapshot_date_key', SNAPSHOT_KEY)
fact_performance['morningstar_rating'] = fact_performance['morningstar_rating'].round().astype('Int64')

print(fact_performance.shape)
print('Repeated amfi_code:', fact_performance['amfi_code'].duplicated().sum())

Snapshot date: 2026-05-29
(40, 16)
Repeated amfi_code: 0


In [13]:
aum_path = glob.glob(os.path.join(RAW_DIR, '*03_aum_by_fund_house*'))[0]
aum = pd.read_csv(aum_path)

print(aum.shape)
print(aum.dtypes)
print(aum['date'].head(10))

(90, 5)
date                  str
fund_house            str
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64
dtype: object
0    2022-03-31
1    2022-03-31
2    2022-03-31
3    2022-03-31
4    2022-03-31
5    2022-03-31
6    2022-03-31
7    2022-03-31
8    2022-03-31
9    2022-03-31
Name: date, dtype: str


In [14]:
AUM_DATE_FORMAT = '%Y-%m-%d'

aum_dates = pd.to_datetime(aum['date'], format=AUM_DATE_FORMAT, errors='coerce')
print('Dates not readable:', (aum_dates.isnull() & aum['date'].notnull()).sum())
assert aum_dates.notnull().all(), 'Fix the AUM dates first'

aum['fund_house'] = aum['fund_house'].str.strip()

ratio = aum['aum_crore'] / aum['aum_lakh_crore']
print(ratio.describe().round(1))

Dates not readable: 0
count        90.0
mean     100000.0
std           0.0
min      100000.0
25%      100000.0
50%      100000.0
75%      100000.0
max      100000.0
dtype: float64


In [15]:
fact_aum = pd.DataFrame({
    'date_key': to_date_key(aum_dates),
    'fund_house': aum['fund_house'],
    'aum_crore': aum['aum_crore'],
    'num_schemes': aum['num_schemes'].round().astype('Int64'),
})

print(fact_aum.shape)
print('Repeated (date_key, fund_house):', fact_aum.duplicated(subset=['date_key', 'fund_house']).sum())
print('Missing fund_house:', fact_aum['fund_house'].isnull().sum())

unknown_houses = set(fact_aum['fund_house']) - set(dim_fund['fund_house'])
print('fund_house names in fact_aum but not in dim_fund:', sorted(unknown_houses))

(90, 4)
Repeated (date_key, fund_house): 0
Missing fund_house: 0
fund_house names in fact_aum but not in dim_fund: []


In [16]:
all_dates = pd.concat([
    nav['date'],
    tx['transaction_date'],
    aum_dates,
    pd.Series([SNAPSHOT_DATE]),
])

START = all_dates.min().normalize()
END = all_dates.max().normalize()

dim_date = build_dim_date(START, END)
n_days = (END - START).days + 1

print('From', START.date(), 'to', END.date())
print('Rows in dim_date:', len(dim_date), '| expected days:', n_days)

From 2022-01-03 to 2026-05-29
Rows in dim_date: 1608 | expected days: 1608


In [17]:
parent_dates = set(dim_date['date_key'])
parent_funds = set(dim_fund['amfi_code'])

fk_checks = [
    ('fact_nav', fact_nav, 'date_key', 'amfi_code'),
    ('fact_transactions', fact_transactions, 'date_key', 'amfi_code'),
    ('fact_performance', fact_performance, 'snapshot_date_key', 'amfi_code'),
]

for name, df, date_col, fund_col in fk_checks:
    bad_dates = (~df[date_col].isin(parent_dates)).sum()
    bad_funds = (~df[fund_col].isin(parent_funds)).sum()
    print(name, '| date_key not in dim_date:', bad_dates, '| amfi_code not in dim_fund:', bad_funds)

print('fact_aum | date_key not in dim_date:', (~fact_aum['date_key'].isin(parent_dates)).sum())

fact_nav | date_key not in dim_date: 0 | amfi_code not in dim_fund: 0
fact_transactions | date_key not in dim_date: 0 | amfi_code not in dim_fund: 0
fact_performance | date_key not in dim_date: 0 | amfi_code not in dim_fund: 0
fact_aum | date_key not in dim_date: 0


In [18]:
LOAD_ORDER = [
    ('dim_date', dim_date),
    ('dim_fund', dim_fund),
    ('fact_nav', fact_nav),
    ('fact_transactions', fact_transactions),
    ('fact_performance', fact_performance),
    ('fact_aum', fact_aum),
]

for name, df in LOAD_ORDER:
    table_cols = pd.read_sql_query(f'PRAGMA table_info({name})', engine)['name'].tolist()
    extra = set(df.columns) - set(table_cols)
    print(name, '| columns that are not in the table:', sorted(extra))

dim_date | columns that are not in the table: []
dim_fund | columns that are not in the table: []
fact_nav | columns that are not in the table: []
fact_transactions | columns that are not in the table: []
fact_performance | columns that are not in the table: []
fact_aum | columns that are not in the table: []


In [19]:
with engine.begin() as conn:
    for name, df in reversed(LOAD_ORDER):
        conn.execute(text(f'DELETE FROM {name}'))

    for name, df in LOAD_ORDER:
        df.to_sql(name, conn, if_exists='append', index=False, chunksize=50_000)
        print('Loaded', name, len(df))

print('Done')

Loaded dim_date 1608
Loaded dim_fund 40
Loaded fact_nav 64320
Loaded fact_transactions 32778
Loaded fact_performance 40
Loaded fact_aum 90
Done


In [20]:
def csv_rows(path):
    try:
        return len(pd.read_csv(path, usecols=[0]))
    except UnicodeDecodeError:
        return len(pd.read_csv(path, usecols=[0], encoding='ISO-8859-1'))


def db_rows(table):
    with engine.connect() as conn:
        return conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()


sources = [
    ('dim_fund', master_path, dim_fund),
    ('fact_nav', nav_path, fact_nav),
    ('fact_transactions', tx_path, fact_transactions),
    ('fact_performance', perf_path, fact_performance),
    ('fact_aum', aum_path, fact_aum),
]

rows = []
for table, path, df in sources:
    source = csv_rows(path)
    in_db = db_rows(table)
    rows.append({'table': table, 'source_rows': source, 'dataframe_rows': len(df),
                 'database_rows': in_db, 'match': source == in_db})

rows.append({'table': 'dim_date', 'source_rows': n_days, 'dataframe_rows': len(dim_date),
             'database_rows': db_rows('dim_date'), 'match': n_days == db_rows('dim_date')})

count_check = pd.DataFrame(rows)
print(count_check.to_string(index=False))

            table  source_rows  dataframe_rows  database_rows  match
         dim_fund           40              40             40   True
         fact_nav        64320           64320          64320   True
fact_transactions        32778           32778          32778   True
 fact_performance           40              40             40   True
         fact_aum           90              90             90   True
         dim_date         1608            1608           1608   True


In [21]:
with engine.connect() as conn:
    db_amount = conn.execute(text('SELECT SUM(amount_inr) FROM fact_transactions')).scalar()
    db_nav_sum = conn.execute(text('SELECT SUM(nav) FROM fact_nav')).scalar()
    db_aum = conn.execute(text('SELECT SUM(aum_crore) FROM fact_aum')).scalar()
    fk_problems = conn.execute(text('PRAGMA foreign_key_check')).fetchall()
    integrity = conn.execute(text('PRAGMA integrity_check')).scalar()

print('Transaction amounts match:', np.isclose(db_amount, fact_transactions['amount_inr'].sum()))
print('NAV sum matches:', np.isclose(db_nav_sum, fact_nav['nav'].sum()))
print('AUM sum matches:', np.isclose(db_aum, fact_aum['aum_crore'].sum()))
print('Foreign key problems:', len(fk_problems))
print('Integrity check:', integrity)

Transaction amounts match: True
NAV sum matches: True
AUM sum matches: True
Foreign key problems: 0
Integrity check: ok


In [22]:
sql = """
SELECT t.year, t.month, f.category, SUM(x.amount_inr) AS total_inr
FROM fact_transactions AS x
JOIN dim_date AS t ON t.date_key = x.date_key
JOIN dim_fund AS f ON f.amfi_code = x.amfi_code
WHERE x.transaction_type = 'SIP'
GROUP BY t.year, t.month, f.category
ORDER BY t.year, t.month, f.category
LIMIT 12
"""

pd.read_sql_query(sql, engine)

,year,month,category,total_inr
0,2024,1,Debt,1825623.0
1,2024,1,Equity,10809726.0
2,2024,2,Debt,1925397.0
3,2024,2,Equity,10687979.0
4,2024,3,Debt,1864273.0
5,2024,3,Equity,10224140.0
6,2024,4,Debt,1898383.0
7,2024,4,Equity,11614002.0
8,2024,5,Debt,2664347.0
9,2024,5,Equity,10554259.0


In [24]:
lines = [
    '# SQLite load summary',
    '',
    f'- Database: bluestock_dw.db. Tables loaded: {len(LOAD_ORDER)}. All loaded in one transaction.',
    '- Method: SQLAlchemy engine and DataFrame.to_sql with if_exists=append. Foreign keys were ON.',
    '',
    '## Row counts (source, DataFrame, database)',
    '```',
    count_check.to_string(index=False),
    '```',
    '',
    '## Other checks',
    f'- Transaction amounts match: {np.isclose(db_amount, fact_transactions["amount_inr"].sum())}.',
    f'- NAV sum matches: {np.isclose(db_nav_sum, fact_nav["nav"].sum())}.',
    f'- AUM sum matches: {np.isclose(db_aum, fact_aum["aum_crore"].sum())}.',
    f'- Foreign key problems: {len(fk_problems)}. Integrity check: {integrity}.',
    f'- fund_house names in fact_aum that are not in dim_fund: {len(unknown_houses)}.',
    f'- dim_date: {n_days} days, from {START.date()} to {END.date()}.',
    '',
]

text_out = '\n'.join(lines)

with open(os.path.join(REPORTS_DIR, 'sqlite_load_summary.md'), 'w', encoding='utf-8') as f:
    f.write(text_out)

print(text_out)

# SQLite load summary

- Database: bluestock_dw.db. Tables loaded: 6. All loaded in one transaction.
- Method: SQLAlchemy engine and DataFrame.to_sql with if_exists=append. Foreign keys were ON.

## Row counts (source, DataFrame, database)
```
            table  source_rows  dataframe_rows  database_rows  match
         dim_fund           40              40             40   True
         fact_nav        64320           64320          64320   True
fact_transactions        32778           32778          32778   True
 fact_performance           40              40             40   True
         fact_aum           90              90             90   True
         dim_date         1608            1608           1608   True
```

## Other checks
- Transaction amounts match: True.
- NAV sum matches: True.
- AUM sum matches: True.
- Foreign key problems: 0. Integrity check: ok.
- fund_house names in fact_aum that are not in dim_fund: 0.
- dim_date: 1608 days, from 2022-01-03 to 2026-05-29.

